In [ ]:
from pathlib import Path

import anndata as ad
import pandas as pd
from abc_atlas_access.abc_atlas_cache.abc_project_cache import AbcProjectCache
from scipy.sparse import csr_matrix

In [ ]:

download_base = Path("/dh-projects/ag-ishaque/analysis/malta/PhD/SL2/SpatialLeiden2-Study/data/ABC_atlas")
data_path = Path("/dh-projects/ag-ishaque/analysis/malta/PhD/SL2/SpatialLeiden2-Study/data/ABC_atlas")

# download data
abc_cache = AbcProjectCache.from_cache_dir(download_base)
abc_cache.load_manifest("releases/20250131/manifest.json")

expression_file = abc_cache.get_data_path(
    directory="Zhuang-ABCA-1", file_name="Zhuang-ABCA-1/raw"
)

cell_file = abc_cache.get_metadata_path(
    directory="Zhuang-ABCA-1", file_name="cell_metadata_with_cluster_annotation"
)

# download CCF annotation
parcellation = abc_cache.get_metadata_dataframe(
    directory="Allen-CCF-2020",
    file_name="parcellation_to_parcellation_term_membership_acronym",
    index_col="parcellation_index",
).astype("category")

parcellation_color = abc_cache.get_metadata_dataframe(
    directory="Allen-CCF-2020",
    file_name="parcellation_to_parcellation_term_membership_color",
    index_col="parcellation_index",
).astype("category")

parcellation.join(parcellation_color).to_csv(data_path / "parcellation.tsv", sep="\t")

abc_cache.get_metadata_dataframe(
    directory="Zhuang-ABCA-1-CCF",
    file_name="ccf_coordinates",
    index_col="cell_label",
    dtype={"cell_label": str},  # FIX: Zell-IDs als String einlesen
).rename(columns={"x": "x_ccf", "y": "y_ccf", "z": "z_ccf"}).to_csv(
    data_path / "CCF.tsv", sep="\t"
)


# prepare anndata object
coord_columns = ["x", "y", "z"]
cell_meta_columns = ["neurotransmitter", "class", "subclass", "supertype", "cluster"]

adata = ad.read_h5ad(expression_file)
adata.X = csr_matrix(adata.X)

cells = pd.read_csv(
    cell_file,
    index_col="cell_label",
    usecols=["cell_label"] + coord_columns + cell_meta_columns,
    dtype={"cell_label": str},
)

adata.obs_names = adata.obs_names.astype(str)
adata = adata[adata.obs_names.isin(cells.index), :]
adata.obs = adata.obs.join(cells)
adata.obsm["spatial"] = adata.obs[coord_columns].to_numpy()
adata.obs = adata.obs.drop(columns=coord_columns).astype("category")
adata.uns["batch_keys"] = ["brain_section_label"]
adata.write_h5ad(data_path / "Zhuang-ABCA-1.h5ad")

In [ ]:

download_base = Path("/dh-projects/ag-ishaque/analysis/malta/PhD/SL2/SpatialLeiden2-Study/data/ABC_atlas")
data_path = download_base
download_base.mkdir(parents=True, exist_ok=True)

# download data
abc_cache = AbcProjectCache.from_cache_dir(download_base)
abc_cache.load_manifest("releases/20250131/manifest.json")
expression_file = abc_cache.get_data_path(
    directory="Zhuang-ABCA-3", file_name="Zhuang-ABCA-3/raw"
)
cell_file = abc_cache.get_metadata_path(
    directory="Zhuang-ABCA-3", file_name="cell_metadata_with_cluster_annotation"
)
# download CCF annotation
parcellation = abc_cache.get_metadata_dataframe(
    directory="Allen-CCF-2020",
    file_name="parcellation_to_parcellation_term_membership_acronym",
    index_col="parcellation_index",
).astype("category")
parcellation_color = abc_cache.get_metadata_dataframe(
    directory="Allen-CCF-2020",
    file_name="parcellation_to_parcellation_term_membership_color",
    index_col="parcellation_index",
).astype("category")
parcellation.join(parcellation_color).to_csv(data_path / "parcellation.tsv", sep="\t")
abc_cache.get_metadata_dataframe(
    directory="Zhuang-ABCA-3-CCF",
    file_name="ccf_coordinates",
    index_col="cell_label",
).rename(columns={"x": "x_ccf", "y": "y_ccf", "z": "z_ccf"}).to_csv(
    data_path / "CCF_Zhuang-ABCA-3.tsv", sep="\t"
)
# prepare anndata object
coord_columns = ["x", "y", "z"]
cell_meta_columns = ["neurotransmitter", "class", "subclass", "supertype", "cluster"]
adata = ad.read_h5ad(expression_file)
adata.X = csr_matrix(adata.X)
cells = pd.read_csv(
    cell_file,
    index_col="cell_label",
    usecols=["cell_label"] + coord_columns + cell_meta_columns,
    dtype={"cell_label": str},
)

adata.obs_names = adata.obs_names.astype(str)
adata = adata[adata.obs_names.isin(cells.index), :]
adata.obs = adata.obs.join(cells)
adata.obsm["spatial"] = adata.obs[coord_columns].to_numpy()
adata.obs = adata.obs.drop(columns=coord_columns).astype("category")
adata.uns["batch_keys"] = ["brain_section_label"]
adata.uns["orientation"] = "sagittal"
adata.write_h5ad(data_path / "Zhuang-ABCA-3.h5ad")

In [ ]:
import anndata as ad
import pandas as pd
from pathlib import Path
from scipy.sparse import csr_matrix
from abc_atlas_access.abc_atlas_cache.abc_project_cache import AbcProjectCache

download_base = Path("/dh-projects/ag-ishaque/analysis/malta/PhD/SL2/SpatialLeiden2-Study/data/ABC_atlas")
data_path = download_base
download_base.mkdir(parents=True, exist_ok=True)

# --- download data ---
abc_cache = AbcProjectCache.from_cache_dir(download_base)
abc_cache.load_manifest("releases/20250131/manifest.json")

# --- CCF parcellation reference (shared across all datasets, only needs downloading once) ---
parcellation = abc_cache.get_metadata_dataframe(
    directory="Allen-CCF-2020",
    file_name="parcellation_to_parcellation_term_membership_acronym",
    index_col="parcellation_index",
).astype("category")
parcellation_color = abc_cache.get_metadata_dataframe(
    directory="Allen-CCF-2020",
    file_name="parcellation_to_parcellation_term_membership_color",
    index_col="parcellation_index",
).astype("category")
parcellation.join(parcellation_color).to_csv(data_path / "parcellation.tsv", sep="\t")

coord_columns = ["x", "y", "z"]
cell_meta_columns = ["neurotransmitter", "class", "subclass", "supertype", "cluster"]

DATASETS = {
    "Zhuang-ABCA-2": "coronal",
    "Zhuang-ABCA-4": "sagittal",
}

for dataset, orientation in DATASETS.items():
    print(f"--- {dataset} ({orientation}) ---")

    expression_file = abc_cache.get_data_path(
        directory=dataset, file_name=f"{dataset}/raw"
    )
    cell_file = abc_cache.get_metadata_path(
        directory=dataset, file_name="cell_metadata_with_cluster_annotation"
    )

    # per-dataset CCF coordinates
    abc_cache.get_metadata_dataframe(
        directory=f"{dataset}-CCF",
        file_name="ccf_coordinates",
        index_col="cell_label",
    ).rename(columns={"x": "x_ccf", "y": "y_ccf", "z": "z_ccf"}).to_csv(
        data_path / f"CCF_{dataset}.tsv", sep="\t"
    )

    # build anndata
    adata = ad.read_h5ad(expression_file)
    adata.X = csr_matrix(adata.X)
    cells = pd.read_csv(
        cell_file,
        index_col="cell_label",
        usecols=["cell_label"] + coord_columns + cell_meta_columns,
        dtype={"cell_label": str},
    )
    adata.obs_names = adata.obs_names.astype(str)
    adata = adata[adata.obs_names.isin(cells.index), :]
    adata.obs = adata.obs.join(cells)
    adata.obsm["spatial"] = adata.obs[coord_columns].to_numpy()
    adata.obs = adata.obs.drop(columns=coord_columns).astype("category")
    adata.uns["batch_keys"] = ["brain_section_label"]
    adata.uns["orientation"] = orientation
    adata.write_h5ad(data_path / f"{dataset}.h5ad")

    print(f"{dataset}: {adata.shape[0]} cells, {adata.shape[1]} genes -> {data_path / f'{dataset}.h5ad'}")